In [ ]:
"""
SiPM IV Curve Analysis Tool

Analyzes current-voltage (IV) characteristics of Silicon Photomultipliers (SiPMs)
and determines breakdown voltages using Landau distribution fitting.

Usage:
    python sipm_IV_curves.py <path_to_folder>

The folder should contain .txt files with voltage and current data.
"""

import os
import re
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.optimize import curve_fit
from scipy.stats import moyal


def first_number_file(name: str) -> int:
    """
    Extract the first number from a filename.

    Args:
        name (str): Filename to parse

    Returns:
        int or None: First number found in the filename, or None if no number exists
    """
    match = re.search(r'\d+', name)
    if match:
        result = int(match.group())
    else:
        result = None
        print(f'Warning: No number found in {name}.')
    return result


def find_key(file: str) -> str:
    """
    Find the first key in an HDF5 file.

    Args:
        file (str): Path to HDF5 file

    Returns:
        str: First key in the HDF5 file
    """
    with h5py.File(file, 'r') as f:
        key = next(iter(f.keys()))
        print("hdf5 keys:", list(f.keys()))
    return key


def natural_key(text: str) -> list:
    """
    Generate a key for natural sorting (handles numbers in strings correctly).

    Args:
        text (str): String to generate sort key for

    Returns:
        list: Sort key that handles numeric and text parts separately

    Example:
        ['file1.txt', 'file10.txt', 'file2.txt'] -> ['file1.txt', 'file2.txt', 'file10.txt']
    """
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', text)]


def plot_IV_overlay(file_names: list, log: bool = False, save_dir: str = None):
    fig, ax = plt.subplots(figsize=(10, 6))

    for file in file_names:
        sipm_n = first_number_file(file)

        with open(file, 'r') as f:
            first_line = f.readline()
            sep = '\t' if '\t' in first_line else (',' if ',' in first_line else ' ')

        df = pd.read_csv(file, sep=sep, header=None, names=["Voltage", "Current"])
        df["Voltage"] = pd.to_numeric(df["Voltage"], errors='coerce')
        df["Current"] = pd.to_numeric(df["Current"], errors='coerce')
        df = df.dropna()

        ax.plot(df["Voltage"], df["Current"], marker='o', markersize=3,
                markevery=20, linewidth=0, label=f"SiPM {sipm_n}")

    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("Current (mA)")
    ax.set_title("IV Curves - All SiPMs")
    if log:
        ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    # SAVE AS PNG
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "IV_overlay.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()


def landau(x: np.array,
           A: float,
           mu: float,
           sigma: float):
    """
    Landau distribution using scipy's Moyal approximation.

    Args:
        x (array): Input values
        A (float): Amplitude parameter
        mu (float): Location parameter (peak position)
        sigma (float): Scale parameter (width)

    Returns:
        array: Landau distribution values
    """
    return A * moyal.pdf(x, loc=mu, scale=sigma)


def analyze_breakdown(file_names: list,
                      window: float = 0.2) -> list:
    """
    Analyze breakdown voltage from IV curves using derivative method.

    The breakdown voltage is identified by fitting a Landau distribution to
    the quantity (1/I) * (dI/dV), which peaks at the breakdown point.

    Args:
        file_names (list): List of file paths containing IV data
        window (float): Half-width (in volts) around the mean for xlim (currently unused)

    Returns:
        list: List of tuples (sipm_id, breakdown_voltage, voltage_error)
    """
    results = []

    for file in file_names:
        # Load data
        sipm_n = first_number_file(file)
        df = pd.read_csv(file, sep="\t")
        df.columns = ["Voltage", "Current"]
        V = df["Voltage"].values
        I = df["Current"].values

        # Calculate derivative dI/dV
        dIdV = np.gradient(I, V)

        # Calculate breakdown indicator: (1/I) * (dI/dV)
        # This quantity peaks sharply at breakdown voltage
        y = (1 / I) * dIdV

        # Remove last two points to avoid edge effects
        V = V[:-2]
        y = y[:-2]

        # Initial parameter guesses for Landau fit
        A_guess = 0  # Amplitude
        mu_guess = V[np.argmax(y)]  # Peak position
        sigma_guess = (np.max(V) - np.min(V)) / 10  # Width estimate

        fit_success = True
        try:
            # Attempt Landau fit
            popt, pcov = curve_fit(
                landau, V, y,
                p0=[A_guess, mu_guess, sigma_guess],
                maxfev=10000)
            A, mu, sigma = popt
            mu_err = np.sqrt(pcov[1, 1])  # Standard error on breakdown voltage
        except Exception:
            # Fallback if fit fails: use peak position with estimated error
            fit_success = False
            mu = V[np.argmax(y)]
            mu_err = 0.03

        results.append((str(sipm_n), mu, mu_err))

        # --- Plot breakdown analysis for this file ---
        plt.figure()
        plt.plot(V, y, 'o', label="(1/I) * (dI/dV)")

        # Overlay Landau fit if successful
        if fit_success:
            V_fit = np.linspace(min(V), max(V), 500)
            plt.plot(V_fit, landau(V_fit, *popt), label="Landau fit")

        # Set plot limits
        plt.xlim(49, 53)
        plt.yscale('log')
        plt.ylim(1e-6, 1e2)

        plt.xlabel("Voltage (V)")
        plt.ylabel("(1/I) * (dI/dV)")
        plt.title(f"Breakdown analysis: {file}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show(block=True)  # Display and wait for user to close
        plt.close()  # Close figure after user interaction

    return results


def plot_breakdown_summary(results: list,
                           ylim: tuple = None):
    """
    Plot summary of breakdown voltages for all analyzed SiPMs.

    Results are sorted by breakdown voltage and displayed with error bars.

    Args:
        results (list): List of tuples (sipm_id, breakdown_voltage, voltage_error)
        ylim (tuple): Optional y-axis limits as (ymin, ymax)
    """
    # Sort results by breakdown voltage (ascending)
    results_sorted = sorted(results, key=lambda r: r[1])

    # Extract sorted values
    names = [r[0] for r in results_sorted]
    mus = [r[1] for r in results_sorted]
    mu_errs = [r[2] for r in results_sorted]

    x = np.arange(len(names))

    # Create plot with error bars
    plt.figure()
    plt.errorbar(x, mus, yerr=mu_errs, fmt='o', capsize=5)

    # Configure x-axis
    plt.xticks(x, names, rotation=45, ha='right')

    if ylim:
        plt.ylim(ylim)

    plt.xlabel("Dataset")
    plt.ylabel("Breakdown Voltage (V)")
    plt.title("Breakdown Voltage (Sorted)")
    plt.grid(True)

    plt.tight_layout()
    plt.show(block=True)  # Display and wait for user to close
    plt.close()  # Close figure after user interaction


def save_raw_data_plots(file_names: list, output_dir: str = None):
    """
    Save raw IV curves as PNG images for each SiPM.

    Args:
        file_names (list): List of file paths containing IV data
        output_dir (str): Directory to save outputs (if None, uses parent folder)
    """
    # Create output directory if it doesn't exist
    if output_dir is None:
        output_dir = os.path.dirname(file_names[0])

    plots_dir = os.path.join(output_dir, "raw_IV_plots")
    os.makedirs(plots_dir, exist_ok=True)

    print(f"\n📸 Saving raw IV plots to: {plots_dir}")

    for file in file_names:
        sipm_id = first_number_file(file)

        # Auto-detect separator and load data
        with open(file, 'r') as f:
            first_line = f.readline()
            sep = '\t' if '\t' in first_line else (',' if ',' in first_line else ' ')

        df = pd.read_csv(file, sep=sep, header=None, names=["Voltage", "Current"])
        df["Voltage"] = pd.to_numeric(df["Voltage"], errors='coerce')
        df["Current"] = pd.to_numeric(df["Current"], errors='coerce')
        df = df.dropna()

        # --- Plot 1: Linear scale ---
        fig1, ax1 = plt.subplots(figsize=(8, 6))
        ax1.plot(df["Voltage"], df["Current"], 'b-', linewidth=1.5)
        ax1.set_xlabel("Voltage (V)", fontsize=12)
        ax1.set_ylabel("Current (mA)", fontsize=12)
        ax1.set_title(f"SiPM {sipm_id} - IV Characteristics", fontsize=14)
        ax1.grid(True, alpha=0.3)
        plt.tight_layout()

        # Save as PNG
        plot_path_linear = os.path.join(plots_dir, f"sipm_{sipm_id}_IV_linear.png")
        plt.savefig(plot_path_linear, dpi=150, bbox_inches='tight')
        plt.close(fig1)
        print(f"   ✓ Saved: sipm_{sipm_id}_IV_linear.png")

        # --- Plot 2: Log scale for current (better for low currents) ---
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        ax2.semilogy(df["Voltage"], df["Current"], 'r-', linewidth=1.5)
        ax2.set_xlabel("Voltage (V)", fontsize=12)
        ax2.set_ylabel("Current (mA) - Log Scale", fontsize=12)
        ax2.set_title(f"SiPM {sipm_id} - IV Characteristics (Log Scale)", fontsize=14)
        ax2.grid(True, alpha=0.3, which='both')
        plt.tight_layout()

        # Save as PNG
        plot_path_log = os.path.join(plots_dir, f"sipm_{sipm_id}_IV_log.png")
        plt.savefig(plot_path_log, dpi=150, bbox_inches='tight')
        plt.close(fig2)
        print(f"   ✓ Saved: sipm_{sipm_id}_IV_log.png")

    # --- Plot 3: Overlay of all SiPMs for comparison ---
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    for file in file_names:
        sipm_id = first_number_file(file)
        with open(file, 'r') as f:
            first_line = f.readline()
            sep = '\t' if '\t' in first_line else (',' if ',' in first_line else ' ')
        df = pd.read_csv(file, sep=sep, header=None, names=["Voltage", "Current"])
        df["Voltage"] = pd.to_numeric(df["Voltage"], errors='coerce')
        df["Current"] = pd.to_numeric(df["Current"], errors='coerce')
        df = df.dropna()
        ax3.plot(df["Voltage"], df["Current"], linewidth=1.5, label=f"SiPM {sipm_id}")

    ax3.set_xlabel("Voltage (V)", fontsize=12)
    ax3.set_ylabel("Current (mA)", fontsize=12)
    ax3.set_title("All SiPMs - IV Comparison", fontsize=14)
    ax3.legend(loc='best')
    ax3.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save overlay plot
    overlay_path = os.path.join(plots_dir, "all_sipms_IV_overlay.png")
    plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
    plt.close(fig3)
    print(f"   ✓ Saved: all_sipms_IV_overlay.png")

    print(f"\n✅ All raw IV plots saved to: {plots_dir}")


def main():
    # CHANGE THIS PATH IF NEEDED
    folder_path = r"C:\Users\leaga\Desktop\Internship UoM\SiPMs IV Curves"

    if not os.path.isdir(folder_path):
        print(f"Error: Folder '{folder_path}' not found")
        return

    # Find all .txt files
    file_names = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.txt')]
    file_names = sorted(file_names, key=natural_key)

    if not file_names:
        print(f"No .txt files found in {folder_path}")
        return

    print(f"Folder: {folder_path}")
    print(f"{len(file_names)} files found:")
    for f in file_names:
        print(f"   - {os.path.basename(f)}")

    # Save raw data plots as PNG
    save_raw_data_plots(file_names, folder_path)

    print("\nGenerating overlay plot...")
    plot_IV_overlay(file_names)

    print("\nAnalyzing breakdown voltages...")
    results = analyze_breakdown(file_names)

    print("\nFINAL RESULTS:")
    for sipm_id, vbd, err in sorted(results, key=lambda x: x[1]):
        print(f"   SiPM {sipm_id}: {vbd:.4f} +/- {err:.4f} V")

    plot_breakdown_summary(results)
    print("\nAnalysis complete!")


def main2():
    """
    Main entry point for the SiPM IV curve analysis tool.

    Parses command-line arguments, loads data files, and generates:
    1. Overlay plot of all IV curves
    2. Individual breakdown analysis plots for each SiPM
    3. Summary plot of all breakdown voltages
    """
    # Parse command-line arguments
    parser = argparse.ArgumentParser(
        description="Analyze SiPM IV curves and determine breakdown voltages",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
        Example:
            python sipm_IV_curves.py /path/to/data/folder

        The folder should contain .txt files with tab-separated voltage and current data.
        """)

    parser.add_argument("path", help="Path to folder containing IV curve data files (.txt)")
    args = parser.parse_args()

    folder_path = args.path

    # Validate folder path
    if not os.path.isdir(folder_path):
        print(f"Error: '{folder_path}' is not a valid directory")
        return 1

    # Find all .txt files and sort naturally
    file_names = [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.endswith('.txt')]

    file_names = sorted(file_names, key=natural_key)

    if not file_names:
        print(f"Error: No .txt files found in '{folder_path}'")
        return 1

    print(f"Found {len(file_names)} data files")
    print("Generating IV overlay plot...")

    # Generate plots
    plot_IV_overlay(file_names)

    print("Analyzing breakdown voltages...")
    results = analyze_breakdown(file_names)

    print("\nBreakdown voltage results:")
    for sipm_id, vbd, vbd_err in results:
        print(f"  SiPM {sipm_id}: {vbd:.3f} ± {vbd_err:.3f} V")

    print("\nGenerating summary plot...")
    plot_breakdown_summary(results)

    print("Analysis complete!")
    return 0


if __name__ == "__main__":
    exit(main())